**Name:** Manu Sharan Kumar  
**Roll No:** ACE080BCT037  

---

# Lab 9: Introduction to PyTorch III
## Task 1 — GPU: Training Time vs Inference Time

This notebook compares model **training time** and **inference time** on **CPU vs GPU**, and explains where the final output tensor is stored (CPU or GPU) after a forward pass.

## 1. Why compare CPU vs GPU?

GPUs are built for massive parallel matrix multiplication, which is exactly what neural network training (forward + backward pass) relies on. CPUs execute operations mostly sequentially (with limited parallelism across a few cores), while GPUs have thousands of cores that can perform many matrix operations simultaneously.

- **Training** involves both a **forward pass** (computing predictions) and a **backward pass** (computing gradients via backpropagation) — both are matrix-heavy, so GPU speedup is usually largest here.
- **Inference** only involves the **forward pass** (no gradient computation), so it's faster than training on either device, but GPU is still typically faster than CPU for larger models/batches.

We will:
1. Build a small neural network.
2. Train it once on CPU, once on GPU, and time both.
3. Run inference (forward pass only) on both devices and time both.
4. Check and explain which device the final output tensor lives on.

In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import time

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

PyTorch version: 2.11.0+cu128
CUDA available: True


In [5]:
# Confirm GPU is attached (only works if a GPU runtime is active)
!nvidia-smi

Thu Jul 30 06:14:58 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   57C    P8             10W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 3. Define the model and data

We use a simple feedforward `nn.Module`, similar to the `NeuralNetwork` class from Part 1 / Part 2 of the PyTorch appendix, but scaled up a bit so the CPU vs GPU timing difference is actually visible (very tiny toy models often run faster on CPU due to GPU transfer/launch overhead).

In [6]:
class NeuralNetwork(nn.Module):
    def __init__(self, num_inputs, num_outputs):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(num_inputs, 500),
            nn.ReLU(),
            nn.Linear(500, 500),
            nn.ReLU(),
            nn.Linear(500, num_outputs),
        )

    def forward(self, x):
        return self.layers(x)

# Dummy dataset just for timing purposes
torch.manual_seed(123)
X_train = torch.randn(5000, 1000)
y_train = torch.randint(0, 10, (5000,))

X_test = torch.randn(1000, 1000)
y_test = torch.randint(0, 10, (1000,))

## 4. Training time — CPU vs GPU

We write one function that trains the model for a few epochs on whichever `device` is passed in. This mirrors the pattern from Part 2 of the appendix (`model.to(device)`, `features, labels = features.to(device), labels.to(device)`), just wrapped in a reusable function with timing added.

**Important:** `torch.cuda.synchronize()` is called before stopping the timer on GPU. GPU operations are launched *asynchronously* — without synchronizing, the timer would stop before the GPU actually finishes, giving a falsely small time.

In [7]:
def train_model(device, num_epochs=5, batch_size=64, lr=0.1):
    torch.manual_seed(123)
    model = NeuralNetwork(num_inputs=1000, num_outputs=10).to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)

    Xd, yd = X_train.to(device), y_train.to(device)

    if device.type == "cuda":
        torch.cuda.synchronize()
    start = time.time()

    model.train()
    for epoch in range(num_epochs):
        for i in range(0, len(Xd), batch_size):
            xb = Xd[i:i+batch_size]
            yb = yd[i:i+batch_size]

            logits = model(xb)
            loss = F.cross_entropy(logits, yb)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    if device.type == "cuda":
        torch.cuda.synchronize()
    elapsed = time.time() - start

    return model, elapsed

In [8]:
# ---- Train on CPU ----
cpu_device = torch.device("cpu")
model_cpu, cpu_train_time = train_model(cpu_device)
print(f"CPU training time: {cpu_train_time:.4f} sec")

CPU training time: 1.6093 sec


In [9]:
# ---- Train on GPU ----
gpu_device = torch.device("cuda")
model_gpu, gpu_train_time = train_model(gpu_device)
print(f"GPU training time: {gpu_train_time:.4f} sec")

GPU training time: 0.8445 sec


In [10]:
# ---- Compare ----
speedup = cpu_train_time / gpu_train_time
print(f"CPU training time: {cpu_train_time:.4f} sec")
print(f"GPU training time: {gpu_train_time:.4f} sec")
print(f"Speedup (CPU time / GPU time): {speedup:.2f}x")

CPU training time: 1.6093 sec
GPU training time: 0.8445 sec
Speedup (CPU time / GPU time): 1.91x


**Observation:** The GPU is typically several times faster than the CPU for this model, because the matrix multiplications inside `nn.Linear` layers are executed in parallel across GPU cores instead of mostly sequentially on the CPU. The speedup grows with model size and batch size — for very small toy models, CPU can sometimes even win, because moving data to the GPU (`.to("cuda")`) has its own fixed overhead.

## 5. Inference time — CPU vs GPU

Inference only needs a **forward pass** — no `loss.backward()`, no `optimizer.step()`. We wrap it in `torch.no_grad()` so PyTorch doesn't build a computation graph for gradients, since we don't need gradients at inference time (this also saves memory and time).

In [11]:
NUM_REPEATS = 50  # number of forward passes used for timing

def run_inference(model, device, num_repeats=NUM_REPEATS):
    model = model.to(device)
    model.eval()
    Xd = X_test.to(device)

    if device.type == "cuda":
        torch.cuda.synchronize()
    start = time.time()

    with torch.no_grad():
        for _ in range(num_repeats):
            output = model(Xd)

    if device.type == "cuda":
        torch.cuda.synchronize()
    elapsed = time.time() - start

    return output, elapsed

In [12]:
# ---- Inference on CPU ----
output_cpu, cpu_infer_time = run_inference(model_cpu, cpu_device)
print(f"CPU inference time ({NUM_REPEATS} forward passes): {cpu_infer_time:.4f} sec")

CPU inference time (50 forward passes): 0.6990 sec


In [13]:
# ---- Inference on GPU ----
output_gpu, gpu_infer_time = run_inference(model_gpu, gpu_device)
print(f"GPU inference time ({NUM_REPEATS} forward passes): {gpu_infer_time:.4f} sec")

GPU inference time (50 forward passes): 0.0356 sec


In [14]:
infer_speedup = cpu_infer_time / gpu_infer_time
print(f"CPU inference time: {cpu_infer_time:.4f} sec")
print(f"GPU inference time: {gpu_infer_time:.4f} sec")
print(f"Speedup (CPU time / GPU time): {infer_speedup:.2f}x")

CPU inference time: 0.6990 sec
GPU inference time: 0.0356 sec
Speedup (CPU time / GPU time): 19.65x


## 6. Where is the final output presented — CPU or GPU?

Whichever `device` the **model** and **input tensor** are moved to (via `.to(device)`) is the device the **output tensor** lives on. PyTorch does not automatically move results back to CPU for you.

This matters practically:
- If the model + inputs are on GPU (`cuda:0`), the output tensor is also on `cuda:0`.
- If you try to use that GPU tensor with NumPy (`.numpy()`) or print/plot it in certain libraries, it will raise an error, because NumPy cannot read GPU memory directly.
- To fix this, you must explicitly bring it back with `.cpu()` before converting, e.g. `output.cpu().numpy()`.

In [15]:
print("Model (CPU) output tensor is on device:", output_cpu.device)
print("Model (GPU) output tensor is on device:", output_gpu.device)

Model (CPU) output tensor is on device: cpu
Model (GPU) output tensor is on device: cuda:0


In [16]:
# Demonstration: converting GPU output to NumPy directly will fail
try:
    output_gpu.numpy()
except TypeError as e:
    print("Error:", e)

# Correct way: move to CPU first
output_gpu_numpy = output_gpu.cpu().numpy()
print("Successfully converted after .cpu():", type(output_gpu_numpy))

Error: can't convert cuda:0 device type tensor to numpy. Use Tensor.cpu() to copy the tensor to host memory first.
Successfully converted after .cpu(): <class 'numpy.ndarray'>


## 7. Summary

| Metric | CPU | GPU |
|---|---|---|
| Training time | Slower — mostly sequential computation | Faster — massively parallel matrix operations |
| Inference time | Slower for large batches | Faster for large batches (small models/batches may show less benefit due to transfer overhead) |
| Output tensor location | `cpu` | `cuda:0` (matches whichever device the model/inputs were moved to) |

**Key takeaways:**
1. GPUs speed up both training and inference because the underlying operations (matrix multiplication, convolution, etc.) parallelize well across GPU cores.
2. Always call `torch.cuda.synchronize()` before stopping a timer on GPU, since CUDA operations run asynchronously.
3. The final output tensor's device always matches the device of the model and input tensors — check with `tensor.device`, and use `.cpu()` before converting to NumPy or plotting.